# Implementace

**FAO MULTIPLEX TRADE NETWORK**

Zde analyzujeme různé typy obchodních vztahů mezi státy, získané z FAO (Organizace pro výživu a zemědělství OSN). Celosvětová síť importu/exportu potravin je ekonomická síť, kde vrstvy představují produkty, uzly jsou státy a hrany v každé vrstvě reprezentují vztahy importu/exportu konkrétního potravinového produktu mezi státy. Data byla získána z FAO a byla vytvořena vícevrstvá síť odpovídající obchodování v roce 2010.  

Multiplexní síť použitá v článku je podmnožinou úplné datové sady, která obsahuje 364 vrstev.

**Reference:**  
M. De Domenico, V. Nicosia, A. Arenas, a V. Latora - "Structural reducibility of multilayer networks" - Nature Communications 2015 6, 6864  
Originální data: [https://manliodedomenico.com/data.php](https://manliodedomenico.com/data.php)

**Formát souborů:**  
layerID nodeID nodeID weight  
364 vrstev Multiplex

**Uzly:** 214  
**Hrany:** 318 346  
**Typ:** Finanční, multiplexní, orientovaná, ohodnocená

## Načtení knihoven

In [14]:
import random
from pathlib import Path
import os

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd

In [15]:
random.seed(42)
np.random.seed(42)

In [16]:
SAVE_DIR = Path("../results/implementation")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

PLOT_DIR = SAVE_DIR / "plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

## Načtení datasetu

In [17]:
layers_path = Path("../data/FAO_Multiplex_Trade/fao_trade_layers.txt")
nodes_path = Path("../data/FAO_Multiplex_Trade/fao_trade_nodes.txt")
edges_path = Path("../data/FAO_Multiplex_Trade/fao_trade_multiplex.edges")

layers_df = pd.read_csv(layers_path, sep='\\s+')
nodes_df = pd.read_csv(nodes_path, sep='\\s+')
edges_df = pd.read_csv(edges_path, sep='\\s+',
                    names=['layerID', 'source', 'target', 'weight'])

## Výpis několika náhodných vrstev

In [18]:
layers_df.sample(10, random_state=13)['layerLabel']

214    Lettuce_and_chicory
251          Oil,_rapeseed
27             Food_wastes
61          Cake,_rapeseed
313                    Rye
330                Vanilla
248     Offals,_liver_duck
101           Cocoa,_paste
73            Eggs,_liquid
345               Rapeseed
Name: layerLabel, dtype: object

In [19]:
summary = pd.DataFrame({
    'Počet uzlů': [nodes_df.shape[0]],
    'Počet vrstev': [layers_df.shape[0]],
    'Počet hran': [edges_df.shape[0]]
})
summary

,Počet uzlů,Počet vrstev,Počet hran
0,214,364,318346


In [20]:
import numpy as np
from sklearn.cluster import AgglomerativeClustering

# 1. Prepare sets of edges for each layer
layer_edges = {
    layer_id: set(
        tuple(row[['source', 'target']])
        for _, row in edges_df[edges_df['layerID'] == layer_id].iterrows()
    )
    for layer_id in layers_df['layerID']
}

# 2. Compute Jaccard distance matrix
layer_ids = list(layer_edges.keys())
n_layers = len(layer_ids)
jaccard_dist = np.zeros((n_layers, n_layers))

for i in range(n_layers):
    for j in range(n_layers):
        if i == j:
            jaccard_dist[i, j] = 0
        else:
            a = layer_edges[layer_ids[i]]
            b = layer_edges[layer_ids[j]]
            intersection = len(a & b)
            union = len(a | b)
            jaccard_dist[i, j] = 1 - intersection / union if union > 0 else 1

# 3. Cluster layers into ~6 groups
clustering = AgglomerativeClustering(n_clusters=5, linkage='complete')
labels = clustering.fit_predict(jaccard_dist)

# 4. Assign cluster labels to layers
layers_df['jacard_cluster'] = labels

# 5. Aggregate edges by cluster
edges_with_cluster = edges_df.merge(layers_df[['layerID', 'jacard_cluster']], on='layerID')
aggregated_edges_jacard = edges_with_cluster.groupby(
    ['source', 'target', 'jacard_cluster']
).agg({'weight': 'sum'}).reset_index()

# Summary of merged Jaccard clusters
jacard_summary = layers_df.groupby('jacard_cluster').agg(
    layer_count=('layerID', 'count'),
    example_labels=('layerLabel', lambda x: ', '.join(x.sample(min(3, len(x)))))
).reset_index()

print(jacard_summary)
print("Number of merged groups (clusters):", layers_df['jacard_cluster'].nunique())

   jacard_cluster  layer_count  \
0               0           44   
1               1           73   
2               2           54   
3               3           91   
4               4          102   

                                      example_labels  
0  Tea,_mate_extracts, Pet_food, Glucose_and_dext...  
1  Fat,_liver_prepared_(foie_gras), Oils,_fats_of...  
2  Skins,_sheep,_dry_salted, Meat,_beef_and_veal_...  
3      Lentils, Nutmeg,_mace_and_cardamoms, Molasses  
4  Meat,_beef,_preparations, Offals,_pigs,_edible...  
Number of merged groups (clusters): 5


c:\Users\vojte\projects\school\metody-analyzy-siti-ii\.venv\Lib\site-packages\sklearn\cluster\_agglomerative.py:584: ClusterWarning: The symmetric non-negative hollow observation matrix looks suspiciously like an uncondensed distance matrix
  out = hierarchy.linkage(X, method=linkage, metric=affinity)


# Redukce na menší počet vrstev

In [21]:
def categorize_layer(layer_name):
    layer_lower = layer_name.lower()
    
    # 1. Animal Products (meat, dairy, eggs, live animals)
    animal_keywords = ['meat', 'cattle', 'pig', 'chicken', 'turkey', 'duck', 'goose', 
                      'sheep', 'goat', 'rabbit', 'horse', 'game', 'offal', 'sausage',
                      'milk', 'cheese', 'butter', 'cream', 'yoghurt', 'whey', 'lactose',
                      'egg', 'honey', 'cattle', 'buffalo', 'camel', 'ass', 'mule',
                      'hide', 'skin', 'tallow', 'lard', 'fat', 'grease', 'bacon', 'ham']
    
    # 2. Beverages (alcoholic and non-alcoholic)
    beverage_keywords = ['beverage', 'coffee', 'tea', 'wine', 'beer', 'cider', 
                        'water', 'juice', 'mate', 'vermouth', 'distilled']
    
    # 3. Grains & Cereals (wheat, rice, corn, etc.)
    grain_keywords = ['wheat', 'rice', 'maize', 'corn', 'barley', 'oat', 'rye',
                     'sorghum', 'millet', 'cereal', 'flour', 'bran', 'bulgur',
                     'malt', 'triticale', 'grain', 'buckwheat']
    
    # 4. Fruits & Vegetables
    fruit_veg_keywords = ['fruit', 'apple', 'apricot', 'avocado', 'banana', 'cherry',
                         'grape', 'lemon', 'lime', 'orange', 'pear', 'peach', 'plum',
                         'strawberry', 'melon', 'pineapple', 'mango', 'papaya', 'kiwi',
                         'date', 'fig', 'cranberry', 'blueberry', 'gooseberry',
                         'vegetable', 'tomato', 'potato', 'carrot', 'cabbage', 'lettuce',
                         'onion', 'garlic', 'leek', 'cucumber', 'eggplant', 'pepper',
                         'asparagus', 'artichoke', 'cauliflower', 'broccoli', 'spinach',
                         'pumpkin', 'squash', 'cassava', 'gherkin', 'gourds', 'plantain']
    
    # 5. Oils, Nuts & Seeds
    oil_nut_keywords = ['oil', 'seed', 'nut', 'almond', 'cashew', 'walnut', 'hazelnut',
                       'pistachio', 'chestnut', 'peanut', 'groundnut', 'copra', 'sesame',
                       'sunflower', 'soybean', 'rapeseed', 'linseed', 'cottonseed',
                       'palm', 'olive', 'coconut', 'safflower', 'castor', 'cake,',
                       'margarine', 'fatty']
    
    # 6. Other Agricultural Products (spices, fibers, tobacco, sugar, etc.)
    # This catches everything else
    
    for keyword in animal_keywords:
        if keyword in layer_lower:
            return 'Animal_Products'
    
    for keyword in beverage_keywords:
        if keyword in layer_lower:
            return 'Beverages'
    
    for keyword in grain_keywords:
        if keyword in layer_lower:
            return 'Grains_Cereals'
    
    for keyword in fruit_veg_keywords:
        if keyword in layer_lower:
            return 'Fruits_Vegetables'
    
    for keyword in oil_nut_keywords:
        if keyword in layer_lower:
            return 'Oils_Nuts_Seeds'
    
    return 'Other_Agricultural'

## Aplikace nových kategorií vrstev na původní data

In [22]:
layers_df['category'] = layers_df['layerLabel'].apply(categorize_layer)

In [23]:
category_counts = layers_df['category'].value_counts()
category_counts

category
Animal_Products       94
Other_Agricultural    83
Fruits_Vegetables     63
Oils_Nuts_Seeds       56
Grains_Cereals        40
Beverages             28
Name: count, dtype: int64

## Kombinace hran a váh podle nových kategorií vrstev

In [ ]:
edges_with_category = edges_df.merge(layers_df[['layerID', 'category']], on='layerID')

aggregated_edges = edges_with_category.groupby(
    ['source', 'target', 'category']
).agg({
    'weight': 'sum',  # Sum weights across all layers in category
}).reset_index()

In [25]:
print(f"Aggregated edges: {len(aggregated_edges):,}")
print(f"Number of categories: {aggregated_edges['category'].nunique()}")

Aggregated edges: 47,226
Number of categories: 6


## Rozdíl počtu hran před a po agregaci

In [26]:
original_edge_count = len(edges_df)
aggregated_edge_count = len(aggregated_edges)
reduction = (original_edge_count - aggregated_edge_count) / original_edge_count * 100

print(f"Original edge count: {original_edge_count:,}")
print(f"Aggregated edge count: {aggregated_edge_count:,}")
print(f"Reduction of edges: {reduction:.2f}%")

Original edge count: 318,346
Aggregated edge count: 47,226
Reduction of edges: 85.17%


## Ukázka několika náhodných hran po agregaci podle kategorií vrstev

In [27]:
aggregated_edges.head(10)

,source,target,category,weight
0,1,2,Beverages,8.0
1,1,2,Fruits_Vegetables,8.0
2,1,2,Grains_Cereals,1.0
3,1,2,Oils_Nuts_Seeds,2.0
4,1,2,Other_Agricultural,42.0
5,1,4,Other_Agricultural,59.0
6,1,5,Other_Agricultural,30.0
7,1,6,Animal_Products,2.0
8,1,6,Beverages,4.0
9,1,6,Fruits_Vegetables,189.0


## Vytvoření `networkx` grafu z agregovaných hran

In [ ]:
category_graphs = {}

# Prepare a mapping from nodeID to nodeLabel for fast lookup
node_id_to_label = nodes_df.set_index('nodeID')['nodeLabel'].to_dict()

for category in aggregated_edges['category'].unique():
    cat_edges = aggregated_edges[aggregated_edges['category'] == category]
    
    G = nx.DiGraph()
    
    # Přidat uzly podle všech zdrojů a cílů v této kategorii
    node_ids = set(cat_edges['source']).union(set(cat_edges['target']))
    for node_id in node_ids:
        label = node_id_to_label.get(node_id, str(node_id))
        G.add_node(node_id, label=label)
    
    # Přidat hrany
    for _, edge in cat_edges.iterrows():
        G.add_edge(edge['source'], edge['target'], weight=edge['weight'])
    
    category_graphs[category] = G

KeyError: 'jacard_cluster'

## Statistiky nově vytvořených grafů podle kategorií vrstev

In [ ]:
stats = []
for category, graph in category_graphs.items():
    weights = [d['weight'] for _, _, d in graph.edges(data=True)]
    mean = np.mean(weights)
    median = np.median(weights)
    q1 = np.percentile(weights, 25)
    q3 = np.percentile(weights, 75)
    
    stats.append({
        'Category': category,
        'Mean': mean,
        'Median': median,
        'Q1': q1,
        'Q3': q3
    })

pd.DataFrame(stats)

,Category,Mean,Median,Q1,Q3
0,Beverages,16165.209837,290.0,34.0,2796.75
1,Fruits_Vegetables,17807.902985,279.0,29.0,2822.00
2,Grains_Cereals,17195.703578,363.5,31.0,3919.75
3,Oils_Nuts_Seeds,29711.285903,312.5,29.0,3574.25
4,Other_Agricultural,27703.422439,683.0,62.0,6451.00
5,Animal_Products,33215.000748,589.0,62.0,5550.00


In [30]:
layers_df

,layerID,layerLabel,jacard_cluster,category
0,1,"Beverages,_non_alcoholic",0,Beverages
1,2,Cream_fresh,4,Animal_Products
2,3,Food_prep_nes,0,Other_Agricultural
3,4,"Cheese,_whole_cow_milk",0,Animal_Products
4,5,Cigarettes,4,Other_Agricultural
...,...,...,...,...
359,360,Rabbits_and_hares,2,Animal_Products
360,361,"Silk-worm_cocoons,_reelable",2,Other_Agricultural
361,362,"Skins,_calve,_wet_salted",2,Animal_Products
362,363,"Rice,_milled/husked",2,Grains_Cereals


In [32]:
# print all jaccard clusters with their layer labels
for cluster_id in sorted(layers_df['jacard_cluster'].unique()):
    cluster_layers = layers_df[layers_df['jacard_cluster'] == cluster_id]['layerLabel'].tolist()
    print(f"Jaccard Cluster {cluster_id}:")
    for layer in cluster_layers:
        print(f"  - {layer}")
    print()

Jaccard Cluster 0:
  - Beverages,_non_alcoholic
  - Food_prep_nes
  - Cheese,_whole_cow_milk
  - Chocolate_products_nes
  - Flour,_wheat
  - Fat,_nes,_prepared
  - Beer_of_barley
  - Chillies_and_peppers,_dry
  - Crude_materials
  - Food_preparations,_flour,_malt_extract
  - Food_wastes
  - Fruit,_prepared_nes
  - Beverages,_distilled_alcoholic
  - Bread
  - Cereals,_breakfast
  - Coffee,_extracts
  - Coffee,_roasted
  - Fruit,_dried_nes
  - Cocoa,_powder_&_cake
  - Fatty_acids
  - Juice,_fruit_nes
  - Pastry
  - Macaroni
  - Pepper_(piper_spp.)
  - Pet_food
  - Glucose_and_dextrose
  - Honey,_natural
  - Nuts,_prepared_(exc._groundnuts)
  - Mixes_and_doughs
  - Oil,_vegetable_origin_nes
  - Oil,_essential_nes
  - Sugar_refined
  - Tea
  - Sugar_confectionery
  - Wine
  - Spices,_nes
  - Sugar,_nes
  - Tea,_mate_extracts
  - Vegetables,_dehydrated
  - Vegetables,_preserved_nes
  - Waters,ice_etc
  - Vegetables,_frozen
  - Vegetables_in_vinegar
  - Vegetables,_preserved,_frozen

Jaccard

# Komunity

# Náhodná procházka

# Vizualizace

## Základní statistky

## Komunity

## Náhodná procházka